# The Strategy Menu
A strategy combines one picking rule, one weighting rule, the market hedge on or off, and fixed or updated estimates. Each rule has its own code file and a short slide deck that describes the rule and its code.

The examples choose a rule by its name on the menu, shown in bold in the tables below.

___

## Picking Rules
A picking rule chooses the basket of tickers at each decision of the trading method. The universe holds 491 stocks and 14 exchange-traded funds (ETFs), without `SPY` and the Treasury bill fund `SGOV`. Each picking file returns the tickers of the basket:

| Rule | What it picks | Code | Slides |
|:--|:--|:--|:--|
| __gamma by sector__ | The three stocks with the highest preference weights in each sector, and the three ETFs with the highest preference weights. | [`gamma_by_sector.jl`](picking/gamma_by_sector.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Gamma-By-Sector-Fall-2026.pdf) |
| __global gamma__ | The 13 tickers with the highest preference weights in the whole universe, with no limit per sector. | [`global_gamma.jl`](picking/global_gamma.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Global-Gamma-Fall-2026.pdf) |
| __correlation gamma__ | Goes down the ranking of preference weights and takes a ticker only if the correlation of its daily growth rates with each ticker already taken is at most 0.7, over the last 252 trading days. If fewer than 13 tickers pass, the rule fills the basket in the order of the ranking. | [`correlation_gamma.jl`](picking/correlation_gamma.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Correlation-Gamma-Fall-2026.pdf) |
| __liquid gamma__ | The 13 tickers with the highest preference weights among the 100 most traded tickers. | [`liquid_gamma.jl`](picking/liquid_gamma.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Liquid-Gamma-Fall-2026.pdf) |
| __hand-picked l8b_13__ and __hand-picked l8b_15__ | The same list of 13 or 15 stocks at every decision, chosen without looking at the 2025 prices. | [`hand_picked.jl`](picking/hand_picked.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Hand-Picked-Fall-2026.pdf) |
| __random by sector__ | Three stocks at random in each sector, and three ETFs at random. | [`random_by_sector.jl`](picking/random_by_sector.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Random-By-Sector-Fall-2026.pdf) |
| __random liquid__ | 13 tickers at random among the 100 most traded tickers. | [`random_liquid.jl`](picking/random_liquid.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Random-Liquid-Fall-2026.pdf) |
| __random global__ | 13 tickers at random from the whole universe. | [`random_global.jl`](picking/random_global.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Random-Global-Fall-2026.pdf) |

The 100 most traded tickers are those with the largest median value traded per day (USD/day) over the last 63 trading days.

The three random rules draw baskets of the same sizes as the gamma rules above them. They show how a basket does by luck alone. Each random rule draws a new basket at each decision, and the saved results run it with 30 seeds in the backtests and 5 in the stress tests.

___

## Weighting Rules
A weighting rule splits the wealth among the tickers of the basket. Each weighting file holds the rule's formula without trading limits:

| Rule | How it splits the wealth | Code | Slides |
|:--|:--|:--|:--|
| __Cobb-Douglas__ | In proportion to the preference weights. A ticker with a preference weight that is not positive gets nothing. | [`cobb_douglas.jl`](weighting/cobb_douglas.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Cobb-Douglas-Fall-2026.pdf) |
| __CES on shares__ | A CES utility on share counts, with an elasticity of 0.5. The weight of each ticker grows with the square root of its price times its preference weight. | [`ces_shares.jl`](weighting/ces_shares.jl) | [slides](slides/CHEME-5660-L15a-Strategy-CES-Shares-Fall-2026.pdf) |
| __CES on weights__ | A CES utility on the weights, with an elasticity of 0.5. The weight of each ticker grows with the square root of its preference weight, which spreads the wealth more evenly than Cobb-Douglas. | [`ces_weights.jl`](weighting/ces_weights.jl) | [slides](slides/CHEME-5660-L15a-Strategy-CES-Weights-Fall-2026.pdf) |
| __Stone-Geary__ | Every ticker gets a baseline weight of 1% of the wealth, and the rest is split as in Cobb-Douglas. | [`stone_geary.jl`](weighting/stone_geary.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Stone-Geary-Fall-2026.pdf) |
| __minimum variance__ | The weights with the smallest variance, with no short sales. | [`minimum_variance.jl`](weighting/minimum_variance.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Minimum-Variance-Fall-2026.pdf) |
| __maximum Sharpe__ | The weights with the largest Sharpe ratio, with no short sales. | [`maximum_sharpe.jl`](weighting/maximum_sharpe.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Maximum-Sharpe-Fall-2026.pdf) |
| __130/30 long-short__ | Cobb-Douglas on each side, with up to 130% of the wealth long and up to 30% short. The long side holds the basket's tickers with positive preference weights, and the short side holds up to three tickers per sector with the most negative preference weights. | [`long_short.jl`](weighting/long_short.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Long-Short-Fall-2026.pdf) |
| __equal weight__ | The same weight on every ticker in the basket. | [`equal_weight.jl`](weighting/equal_weight.jl) | [slides](slides/CHEME-5660-L15a-Strategy-Equal-Weight-Fall-2026.pdf) |
| __BAB__ | Betting against beta (Frazzini and Pedersen): the half of the basket with the lowest market sensitivity goes long, and the half with the highest goes short. Each side is scaled to a market sensitivity of one, so that the market sensitivity of the portfolio is zero before the trading limits. | [`bab.jl`](weighting/bab.jl) | [slides](slides/CHEME-5660-L15a-Strategy-BAB-Fall-2026.pdf) |

The trading method adjusts each formula to meet the trading limits: a limit on the weight of each ticker, a limit on the volatility of the portfolio, and a limit on the trading costs of each decision. The cash may not go below zero. The BAB rule also holds at most 130% long and 30% short.

___

## The Market Hedge and the Estimates
The last two choices are settings of the trading method, not code files:

| Choice | Settings | What it does |
|:--|:--|:--|:--|
| __market hedge__ | `true` (on) or `false` (off) | With the hedge on, the market sensitivity of the portfolio must stay at or below a limit, and the limit falls as the market falls. To meet a low limit, the method can buy up to five tickers whose market sensitivity is zero or negative, hold cash, or sell `SPY` short after a steep fall. |
| __estimates__ | `"fixed"` or `"updated"` | Fixed estimates keep the single index model fitted on 2016 to 2024 for the whole run. Updated estimates start from that fit and update it every trading day with exponentially weighted least squares, so that the weight of a day halves every 63 trading days. |

With the two hand-picked lists, the menu has 9 picking entries and 9 weighting rules. Each pair runs with the hedge on and off, and with fixed and updated estimates, which makes 324 strategies.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success.

Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___